<font size="10">Alpaca VHH Sequencing Part I </font>
    
<font size="5">Raw Read Assembly (pRESTO) </font>
    
Data Input \
Sample - Unique sample ID from raw sequencing read data \
Sub0 -  Directory with raw .fastq files \

Data Output
Sub1 - Directory for intermediate files \
Sub2 - Directory for final output files (.fasta for downstream processing, .fastq to preserve quality scores).  Sub folders for logs and fastq files created automatically 

Sequences are split into duplicate reads from which consensus sequences are produced and single read sequences that are not used


In [7]:
%%bash 
DATA_DIR=/home/troyhinkley/alpaca/EVD68


#for Sample in {A-S1,B-S3,C-S5,D-S7,E-S9,F-S11,D-S2,A-S4,B-S6,E-S8,F-S10}; do \
#for Sample in {10_S1,11_S2,12_S3,13_S4,14_S5,15_S6,16_S7,17_S8,18_S9,19_S10,20_S11,21_S12,22_S13,23_S14,24_S15,25_S16}; do \
#for Sample in {15_S6,16_S7,17_S8,18_S9,19_S10,20_S11,21_S12,22_S13,23_S14,24_S15,25_S16}; do \
#for Sample in {SSA-S1,SA-S1}; do \
#for Sample in {25_S16,PhageDisplay,Alpaca}; do \
#for Sample in {25_S16,PhageDisplay,Alpaca,Gblock,Alex,Lara}; do \


for Sample in {A2A_S1,L2A_S2,PDA_S3,GblockA_S4,NegA_S5,A2G1_S6,L2G1_S7,PD1G1_S8,PD2G1_S9,GblockG1_S10,GloveG1_S11,A2G2_S12,L2G2_S13,PD1G2_S14,GblockG2_S15}; do \
#for Sample in {NegA_S5,A2G1_S6,L2G1_S7}; do \




#Sample=L2G1_S7

#SUBDIRECTORIES
#raw data directory
Sub0=fastq
#intermediate data directory name
Sub1=tmp
#final output name
Sub2=EVD68out

#check if intermediate and final output directories exist and creates them if they do not
if [ ! -d $DATA_DIR/$Sub1 ]; then
    mkdir $DATA_DIR/$Sub1
fi

if [ ! -d $DATA_DIR/$Sub2 ]; then
    mkdir $DATA_DIR/$Sub2
fi


For="$Sample"_L001_R1_001
Rev="$Sample"_L001_R2_001
SEQF=data/$Sub0/$For.fastq
SEQR=data/$Sub0/$Rev.fastq
SEQ1=data/$Sub1/AssemblePairs$Sample.fastq
SEQ2=data/$Sub1/FilterSeq$Sample.fastq
SEQ3=data/$Sub1/MaskPrimersFor$Sample.fastq
SEQ4=data/$Sub1/MaskPrimersRev$Sample.fastq
SEQ5=data/$Sub1/ParseHeaders$Sample.fastq
SEQ6=data/$Sub1/CollapseSeq$Sample.fastq


#assemble paired end reads
sudo docker run -v $DATA_DIR:/data:z immcantation/suite:devel \
    AssemblePairs.py align -1 $SEQF -2 $SEQR --maxerror 0.1 \
        --coord illumina  -o $SEQ1 --rc tail 

#throw out low quality reads, anything below Q30 gets tossed
sudo docker run -v $DATA_DIR:/data:z immcantation/suite:devel \
    FilterSeq.py quality -s $SEQ1 -q 30 -o $SEQ2 

#match the forward primers and read the forward half of the barcode
sudo docker run -v $DATA_DIR:/data:z immcantation/suite:devel \
    MaskPrimers.py align -s $SEQ2 -p data/$Sub0/AlpVHH.fasta --barcode \
         --maxerror 0.01 --mode trim --pf VPRIMER \
        -o $SEQ3 

#match the reverse primers and read the reverse half of the barcode
sudo docker run -v $DATA_DIR:/data:z immcantation/suite:devel \
    MaskPrimers.py align -s $SEQ3 -p data/$Sub0/AlpVHH.fasta --barcode \
         --maxerror 0.01 --mode trim --revpr --pf CPRIMER \
        -o $SEQ4 
            
#concatenate the barcodes from each end to form full unique identifiers
sudo docker run -v $DATA_DIR:/data:z immcantation/suite:devel \
    ParseHeaders.py collapse -s $SEQ4 -f BARCODE --act cat -o $SEQ5
            
# #remove duplicate sequences and annotate with number of identical reads with DUPCOUNT
# sudo docker run -v $DATA_DIR:/data:z immcantation/suite:devel \
#     CollapseSeq.py -s $SEQ5 -n 20 --inner --act set set -o $SEQ6 \
#          --cf VPRIMER CPRIMER
    
# #split sequences by single reads and duplicates, output as .fastq files to keep quality scores
# #sudo docker run -v $DATA_DIR:/data:z immcantation/suite:devel \
# #    SplitSeq.py group -s $SEQ6 -f DUPCOUNT --num 2 \
# #        --outname $Sample --outdir data/$Sub2/fastq

# #split sequences by single reads and duplicates, output as .fasta for repertoire analysis 
# sudo docker run -v $DATA_DIR:/data:z immcantation/suite:devel \
#     SplitSeq.py group -s $SEQ6 -f DUPCOUNT --num 2 --fasta \
#         --outname $Sample --outdir data/$Sub2

done

       START> AssemblePairs
     COMMAND> align
       FILE1> A2A_S1_L001_R1_001.fastq
       FILE2> A2A_S1_L001_R2_001.fastq
  COORD_TYPE> illumina
       ALPHA> 1e-05
   MAX_ERROR> 0.1
     MIN_LEN> 8
     MAX_LEN> 1000
SCAN_REVERSE> False
       NPROC> 16

PROGRESS> 22:51:42 |####################| 100% (2,378,908) 78.3 min

OUTPUT> AssemblePairsA2A_S1.fastq
 PAIRS> 2378908
  PASS> 2099952
  FAIL> 278956
   END> AssemblePairs

   START> FilterSeq
 COMMAND> quality
    FILE> AssemblePairsA2A_S1.fastq
   INNER> False
MIN_QUAL> 30.0
   NPROC> 16

PROGRESS> 22:55:58 |####################| 100% (2,099,952) 4.1 min

   OUTPUT> FilterSeqA2A_S1.fastq
SEQUENCES> 2099952
     PASS> 2087922
     FAIL> 12030
      END> FilterSeq

        START> MaskPrimers
      COMMAND> align
     SEQ_FILE> FilterSeqA2A_S1.fastq
  PRIMER_FILE> AlpVHH.fasta
         MODE> trim
    MAX_ERROR> 0.01
      MAX_LEN> 50
   REV_PRIMER> False
      SKIP_RC> False
  GAP_PENALTY> 1, 1
      BARCODE> True
BARCODE_FIELD> BA

ERROR> Sequence file data/fastq/GloveG1_S11_L001_R1_001.fastq does not exist.

ERROR> Sequence file data/tmp/AssemblePairsGloveG1_S11.fastq does not exist.

ERROR> Sequence file data/tmp/FilterSeqGloveG1_S11.fastq does not exist.

ERROR> Sequence file data/tmp/MaskPrimersForGloveG1_S11.fastq does not exist.

ERROR> Sequence file data/tmp/MaskPrimersRevGloveG1_S11.fastq does not exist.



In [33]:
%%bash


Sample=SSA-S1
SEQ1=data/fullalpaca1/"$Sample"_atleast-2.fasta
SEQ2=data/fullalpaca1/"$Sample"-table.tsv
SEQ3=data/fullalpaca1/"$Sample"-done.tsv


sudo docker run -v $(pwd):/data:z immcantation/suite:devel \
    ParseHeaders.py table -s $SEQ1 -f ID CPRIMER VPRIMER DUPCOUNT -o $SEQ2

sudo docker run -v $(pwd):/data:z immcantation/suite:devel \
    ParseDb.py index -d $SEQ2 -o $SEQ3 -f ID
    
    

  START> ParseHeaders
COMMAND> table
   FILE> SSA-S1_atleast-2.fasta

PROGRESS> 23:18:11 |####################| 100% (275) 0.0 min

   OUTPUT> SSA-S1-table.tsv
SEQUENCES> 275
     PASS> 275
     FAIL> 0
      END> ParseHeaders

  START> ParseDb
COMMAND> index
   FILE> SSA-S1-table.tsv
  FIELD> ID

PROGRESS> 23:18:12 |####################| 100% (275) 0.0 min

 OUTPUT> SSA-S1-done.tsv
RECORDS> 275
    END> ParseDb



WARNING> Output file data/fullalpaca1/SSA-S1-table.tsv already exists and will be overwritten.
WARNING> Output file data/fullalpaca1/SSA-S1-done.tsv already exists and will be overwritten.


In [51]:
%%bash 
DATA_DIR=/home/troyhinkley/alpaca


#for Sample in {A-S1,B-S3,C-S5,D-S7,E-S9,F-S11,D-S2,A-S4,B-S6,E-S8,F-S10}; do \
#for Sample in {10_S1,11_S2,12_S3,13_S4,14_S5,15_S6,16_S7,17_S8,18_S9,19_S10,20_S11,21_S12,22_S13,23_S14,24_S15,25_S16}; do \
#for Sample in {15_S6,16_S7,17_S8,18_S9,19_S10,20_S11,21_S12,22_S13,23_S14,24_S15,25_S16}; do \
#for Sample in {SSA-S1,SA-S1}; do \
Sample=SA-S1


#SUBDIRECTORIES
#raw data directory
Sub0=fastq01
#intermediate data directory
Sub1=int01
#final output directory
Sub2=fullalpacaBC

#check if intermediate and final output directories exist and creates them if they do not
if [ ! -d $DATA_DIR/$Sub1 ]; then
    mkdir $DATA_DIR/$Sub1
fi

if [ ! -d $DATA_DIR/$Sub2 ]; then
    mkdir $DATA_DIR/$Sub2
fi


For="$Sample"_R1
Rev="$Sample"_R2
SEQF=data/$Sub0/$For.fastq
SEQR=data/$Sub0/$Rev.fastq
SEQ1=data/$Sub1/AssemblePairs.fastq
SEQ2=data/$Sub1/FilterSeq.fastq
SEQ3=data/$Sub1/MaskPrimersFor.fastq
SEQ4=data/$Sub1/MaskPrimersRev.fastq
SEQ5=data/$Sub1/ParseHeaders.fastq
SEQ6=data/$Sub1/$Sample-BC1.fastq


#assemble paired end reads
sudo docker run -v $DATA_DIR:/data:z immcantation/suite:devel \
    AssemblePairs.py align -1 $SEQF -2 $SEQR --maxerror 0.1 \
        --coord illumina  -o $SEQ1 --rc tail 

#throw out low quality reads, anything below Q30 gets tossed
sudo docker run -v $DATA_DIR:/data:z immcantation/suite:devel \
    FilterSeq.py quality -s $SEQ1 -q 30 -o $SEQ2 

#match the forward primers and read the forward half of the barcode
sudo docker run -v $DATA_DIR:/data:z immcantation/suite:devel \
    MaskPrimers.py score -s $SEQ2 -p data/$Sub0/AlpVHH.fasta --barcode \
        --start 8 --maxerror 0.01 --mode cut --pf VPRIMER \
        -o $SEQ3 

#match the reverse primers and read the reverse half of the barcode
sudo docker run -v $DATA_DIR:/data:z immcantation/suite:devel \
    MaskPrimers.py score -s $SEQ3 -p data/$Sub0/AlpVHH.fasta --barcode \
        --start 8 --maxerror 0.01 --mode cut --revpr --pf CPRIMER \
        -o $SEQ4 
            
#concatenate the barcodes from each end to form full unique identifiers
sudo docker run -v $DATA_DIR:/data:z immcantation/suite:devel \
    ParseHeaders.py collapse -s $SEQ4 -f BARCODE --act cat -o $SEQ5

sudo docker run -v $DATA_DIR:/data:z immcantation/suite:devel \
    BuildConsensus.py -s $SEQ5  --pf VPRIMER \
         --maxerror 0.1  -o $SEQ6 -n 2 --bf BARCODE




done

       START> AssemblePairs
     COMMAND> align
       FILE1> SA-S1_R1.fastq
       FILE2> SA-S1_R2.fastq
  COORD_TYPE> illumina
       ALPHA> 1e-05
   MAX_ERROR> 0.1
     MIN_LEN> 8
     MAX_LEN> 1000
SCAN_REVERSE> False
       NPROC> 16

PROGRESS> 01:13:00 |####################| 100% (50,000) 1.5 min

OUTPUT> AssemblePairs.fastq
 PAIRS> 50000
  PASS> 47943
  FAIL> 2057
   END> AssemblePairs

   START> FilterSeq
 COMMAND> quality
    FILE> AssemblePairs.fastq
   INNER> False
MIN_QUAL> 30.0
   NPROC> 16

PROGRESS> 01:13:07 |####################| 100% (47,943) 0.1 min

   OUTPUT> FilterSeq.fastq
SEQUENCES> 47943
     PASS> 47616
     FAIL> 327
      END> FilterSeq

        START> MaskPrimers
      COMMAND> score
     SEQ_FILE> FilterSeq.fastq
  PRIMER_FILE> AlpVHH.fasta
         MODE> cut
    MAX_ERROR> 0.01
    START_POS> 8
   REV_PRIMER> False
      BARCODE> True
BARCODE_FIELD> BARCODE
 PRIMER_FIELD> VPRIMER
        NPROC> 16

PROGRESS> 01:13:16 |####################| 100% (47,616) 0.

WARNING> Output file data/int01/AssemblePairs.fastq already exists and will be overwritten.
WARNING> Output file data/int01/FilterSeq.fastq already exists and will be overwritten.
WARNING> Output file data/int01/MaskPrimersFor.fastq already exists and will be overwritten.
WARNING> Output file data/int01/MaskPrimersRev.fastq already exists and will be overwritten.
WARNING> Output file data/int01/ParseHeaders.fastq already exists and will be overwritten.
